In [1]:
# Python imports and pre-definitions
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['font.size'] = 30

def parse_lammps_rdf(rdffile):
    """Parse the RDF file written by LAMMPS
    copied from Boris' class code: https://github.com/bkoz37/labutil
    """
    with open(rdffile, 'r') as rdfout:
        rdfs = []; buffer = []
        for line in rdfout:
            values = line.split()
            if line.startswith('#'):
                continue
            elif len(values) == 2:
                nbins = values[1]
            else:
                buffer.append([float(values[1]), float(values[2])])
                if len(buffer) == int(nbins):
                    frame = np.transpose(np.array(buffer))
                    rdfs.append(frame)
                    buffer = []
    return rdfs

In [2]:
%%capture
# download data
!gdown --folder --id --no-cookies https://drive.google.com/drive/folders/1FEwF4i8IDHGmAIQ3RilA0jG9_lEX4Yk0?usp=sharing
#!mkdir Si_data
#!mv *.xyz ./Si_data

In [3]:
# set to allow anonymous WandB
import os
os.environ["WANDB_ANONYMOUS"] = "must"

In [4]:
import torch

torch.cuda.is_available()

True

In [5]:
!rm -rf ./results

!/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/bin/nequip-train ./config/NaBr.yaml --equivariance-test

[W418 14:25:37.138441069 init.cpp:809] Warning: nvfuser is no longer supported in torch script, use _jit_nvfuser_enabled is deprecated and a no-op (function operator())
[W418 14:25:37.138466518 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
[W418 14:25:37.138475927 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: wandb version 0.19.9 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.16.5
wandb: Run data is saved locally in /pscratch/sd/v/vladygin/NaBr_project/ALLEGRO_setup/new_setup/SpinGNN/allegro/workdir/NaBr/wandb/run-20250418_142545-d05Dy3F3-BX1HEBfkx7fobgqS7o_1ZBsXmU2Ho_yuOM
wandb: Run `wandb offline` to turn off syncing.
wandb: Sync

In [6]:
!/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/bin/nequip-benchmark ./config/NaBr.yaml

Using device: cuda
[W418 14:27:04.737636547 init.cpp:809] Warning: nvfuser is no longer supported in torch script, use _jit_nvfuser_enabled is deprecated and a no-op (function operator())
[W418 14:27:04.737661485 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
[W418 14:27:04.737669150 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
Loading dataset... 
    loading dataset took 0.0182s
    loaded dataset of size 210 and sampled --n-data=2 frames
    benchmark frames statistics:
         number of atoms: 64
         number of types: 2
          avg. num edges: 1221.0
         avg. neigh/atom: 19.078125
Building model... 
Is ij diagonal
tensor(False)
/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript t

In [7]:
!/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/bin/nequip-evaluate --train-dir results/NaBr-tutorial/NaBr --batch-size 1

Traceback (most recent call last):
  File "/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/bin/nequip-evaluate", line 8, in <module>
    sys.exit(main())
  File "/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/nequip/scripts/evaluate.py", line 196, in main
    trainer = torch.load(
  File "/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/serialization.py", line 1425, in load
    with _open_file_like(f, "rb") as opened_file:
  File "/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/serialization.py", line 751, in _open_file_like
    return _open_file(name_or_buffer, mode)
  File "/pscratch/sd/v/vladygin/doped-Si_project/MLFF_TDEP/testbench/lib/python3.10/site-packages/torch/serialization.py", line 732, in __init__
    super().__init__(open(name, mode))
FileNotFoundError: [Errno 2] No such file or directory: 'results/NaBr-tutorial/NaBr/traine

In [9]:
!../allegro_env/bin/nequip-deploy build --train-dir results/NaBr-tutorial/NaBr NaBr-deployed.pth
!ls *pth

INFO:root:Loading best_model from training session...
INFO:root:Compiled & optimized model.
NaBr-deployed.pth


In [3]:
from ase.io import read, write

example_atoms = read('./NaBr_data/NaBr-data.xyz', index=1)
write('./NaBr.data', example_atoms, format='lammps-data')

In [13]:
lammps_input = """
units	metal
atom_style atomic
dimension 3

# set newton on for pair_allegro (off for pair_nequip)
newton on
boundary p p p
read_data ../si.data

# if you want to run a larger system, simply replicate the system in space
# replicate 3 3 3

# allegro pair style
pair_style	allegro
pair_coeff	* * ../si-deployed.pth Si

mass 1 28.0855 

velocity all create 300.0 1234567 loop geom

neighbor 1.0 bin
neigh_modify delay 5 every 1

timestep 0.001
thermo 10

# nose-hoover thermostat, 300K
fix  1 all nvt temp 300 300 $(100*dt)

# compute rdf and average after some equilibration
comm_modify cutoff 7.0
compute rdfall all rdf 1000 cutoff 5.0
fix 2 all ave/time 1 2500 5000 c_rdfall[*] file si.rdf mode vector

# run 5ps
run 5000
"""  
!rm -rf ./lammps_run  
!mkdir lammps_run
with open("lammps_run/si_rdf.in", "w") as f:
    f.write(lammps_input)

In [21]:
!module load PrgEnv-gnu cray-mpich cudatoolkit craype-accel-nvidia80 && cd lammps_run/ && ../../lammps/build/lmp -in si_rdf.in

MPICH ERROR [Rank 0] [job id ] [Tue May 14 14:47:27 2024] [nid001284] - Abort(-1) (rank 0 in comm 0): MPIDI_CRAY_init: GPU_SUPPORT_ENABLED is requested, but GTL library is not linked
 (Other MPI error)

aborting job:
MPIDI_CRAY_init: GPU_SUPPORT_ENABLED is requested, but GTL library is not linked

